In [1]:
import json
import time
import requests
import pandas as pd
import kagglehub
# %pip install transformers
from transformers import AutoTokenizer
from recognition_evaluators import evaluate_tm_df

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kzvau\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
multiclass_dataset_path = kagglehub.dataset_download("sagarikashreevastava/cognitive-distortion-detetction-dataset")
multiclass_dataset_file_path = multiclass_dataset_path + "/Annotated_data.csv"

df = pd.read_csv(multiclass_dataset_file_path)
df = df.drop("Id_Number", axis=1)

held_fraction = 5
train_df = df[df.index % held_fraction != 0].reset_index(drop=True)
held_df = df[df.index % held_fraction == 0].reset_index(drop=True)

df

,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning
...,...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,NaN,No Distortion,NaN
2526,I am 21 female and have not had any friends fo...,Now I am at university my peers around me all ...,Overgeneralization,NaN
2527,From the U.S.: My brother is 19 years old and ...,He claims he’s severely depressed and has outb...,Mental filter,Mind Reading
2528,From the U.S.: I am a 21 year old woman who ha...,NaN,No Distortion,NaN


In [3]:
def choose_text(row):
    if pd.notna(row["Distorted part"]):
        return row["Distorted part"]
    return row["Patient Question"]

def collect_labels(row):
    labels = []
    dominant = row["Dominant Distortion"]
    if dominant != "No Distortion":
        labels.append(dominant)
    secondary = row["Secondary Distortion (Optional)"]
    if pd.notna(secondary):
        if secondary != "No Distortion" and secondary not in labels:
            labels.append(secondary)
    return labels

train_data = pd.DataFrame()
train_data["text"] = train_df.apply(choose_text, axis=1)
train_data["labels"] = train_df.apply(collect_labels, axis=1)
train_data["dominant_label"] = train_df["Dominant Distortion"]

held_data = pd.DataFrame()
held_data["text"] = held_df.apply(choose_text, axis=1)
held_data["labels"] = held_df.apply(collect_labels, axis=1)

all_labels = [
    "All-or-nothing thinking",
    "Emotional Reasoning",
    "Fortune-telling",
    "Labeling",
    "Magnification",
    "Mental filter",
    "Mind Reading",
    "Overgeneralization",
    "Personalization",
    "Should statements"]

train_data

,text,labels,dominant_label
0,I feel trapped inside my disgusting self and l...,"[Labeling, Emotional Reasoning]",Labeling
1,So I’ve been dating on and off this guy for a...,[],No Distortion
2,My parents got divorced in 2004. My mother has...,[],No Distortion
3,I refused to go because I didn’t know if it wa...,"[Fortune-telling, Emotional Reasoning]",Fortune-telling
4,"About a year ago to the month, I was in the mi...",[],No Distortion
...,...,...,...
2019,From the U.S.: I’m a 12th grader in high schoo...,[],No Distortion
2020,Now I am at university my peers around me all ...,[Overgeneralization],Overgeneralization
2021,He claims he’s severely depressed and has outb...,"[Mental filter, Mind Reading]",Mental filter
2022,From the U.S.: I am a 21 year old woman who ha...,[],No Distortion


In [4]:
def build_balanced_train(train_data):
    label_order = sorted(train_data["dominant_label"].unique())
    label_groups = {}
    label_positions = {}

    for label in label_order:
        label_groups[label] = train_data[train_data["dominant_label"] == label].sample(frac=1, random_state=42).reset_index(drop=True)
        label_positions[label] = 0

    balanced_rows = []

    while len(balanced_rows) < len(train_data):
        for label in label_order:
            if label_positions[label] < len(label_groups[label]):
                balanced_rows.append(label_groups[label].iloc[label_positions[label]])
                label_positions[label] += 1

    return pd.DataFrame(balanced_rows).reset_index(drop=True)

balanced_train = build_balanced_train(train_data)

balanced_train.head(15)

,text,labels,dominant_label
0,"Then, she turned the same volatile screaming o...",[All-or-nothing thinking],All-or-nothing thinking
1,ive had a deep seeded hate and anger with me f...,[Emotional Reasoning],Emotional Reasoning
2,I feel I am useless to my kids or my husband b...,[Fortune-telling],Fortune-telling
3,I also tried to cut myself three times in my w...,[Labeling],Labeling
4,"I’m lost in life, I often feel rage, anger and...","[Magnification, Labeling]",Magnification
5,Everyday I feel my life is being wasted.,"[Mental filter, Magnification]",Mental filter
6,They seem not to love and support me.,[Mind Reading],Mind Reading
7,I am seventeen and I dated this guy for about ...,[],No Distortion
8,I also am very unorganized and messy and I str...,[Overgeneralization],Overgeneralization
9,Sometimes i get awful thoughts that makes me f...,"[Personalization, Mind Reading]",Personalization


In [5]:
def labels_to_json(labels):
    return json.dumps(labels, ensure_ascii=False)

def build_many_shot_prefix(train_df, allowed_labels):
    parts = []
    parts.append(
        "You are a professional psychotherapist experienced in cognitive-behavioral therapy.\n"
        "Your task is multi-label classification of cognitive distortions in patient texts.\n"
        "You can assign none, one, or several cognitive distortion labels to each text.\n"
        "Allowed labels are:")

    parts.append(labels_to_json(allowed_labels))

    parts.append(
        "Important annotation rule: if there is no cognitive distortion, return an empty JSON array: [].\n"
        "Do not use the label 'No Distortion'. Absence of distortion must be represented only as [].\n"
        "Return only a valid JSON array of strings. Do not return explanations, comments, markdown, or extra text.\n"
        "Below are annotated examples.")

    for i, row in train_df.iterrows():
        parts.append(f"\nExample {i + 1}:")
        parts.append(f"Text: {row['text']}")
        parts.append(f"Labels: {labels_to_json(row['labels'])}")

    parts.append("\nNow classify new texts using the same label set and the same annotation logic.")
    return "\n".join(parts)

def build_test_prompt(many_shot_prefix, text):
    return many_shot_prefix + "\n\nText to classify:\n" + text + "\n\nReturn only the JSON array of labels:"

In [6]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:7b"
TOKENIZER_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

model_details = requests.post(OLLAMA_BASE_URL + "/api/show", json={"model": MODEL_NAME}).json()
context_values = []

for key, value in model_details["model_info"].items():
    if key.endswith(".context_length"):
        context_values.append(value)

model_context_limit = max(context_values)

longest_held_text = held_data.iloc[0]["text"]
longest_held_tokens = len(tokenizer.encode(longest_held_text))

for _, row in held_data.iterrows():
    text_tokens = len(tokenizer.encode(row["text"]))
    if text_tokens > longest_held_tokens:
        longest_held_text = row["text"]
        longest_held_tokens = text_tokens

print(
    f"Model: {MODEL_NAME}\n"
    f"Parameter size: {model_details['details']['parameter_size']}\n"
    f"Quantization: {model_details['details']['quantization_level']}\n"
    f"Model context limit: {model_context_limit}\n"
    f"Longest held-out text tokens: {longest_held_tokens}"
)

Model: qwen2.5:7b
Parameter size: 7.6B
Quantization: Q4_K_M
Model context limit: 32768
Longest held-out text tokens: 494


In [7]:
MAX_OUTPUT_TOKENS = 128
CONTEXT_RESERVE_TOKENS = 64

def test_example_count(example_count):
    candidate_train = balanced_train.head(example_count)
    candidate_prefix = build_many_shot_prefix(candidate_train, all_labels)
    candidate_prompt = build_test_prompt(candidate_prefix, longest_held_text)
    candidate_context_size = len(tokenizer.encode(candidate_prompt)) + MAX_OUTPUT_TOKENS + CONTEXT_RESERVE_TOKENS

    if candidate_context_size > model_context_limit:
        return False, candidate_context_size

    requests.post(OLLAMA_BASE_URL + "/api/generate", json={"model": MODEL_NAME, "stream": False, "keep_alive": 0}, timeout=3600)

    requests.post(
        OLLAMA_BASE_URL + "/api/generate",
        json={
            "model": MODEL_NAME,
            "prompt": candidate_prompt,
            "stream": False,
            "keep_alive": -1,
            "options": {
                "temperature": 0,
                "num_ctx": candidate_context_size,
                "num_predict": 1}},
        timeout=3600).json()

    running_models = requests.get(OLLAMA_BASE_URL + "/api/ps", timeout=3600).json()["models"]
    running_model = [model for model in running_models if model["name"] == MODEL_NAME or model["model"] == MODEL_NAME][0]
    is_full_gpu = running_model["size_vram"] == running_model["size"]

    return is_full_gpu, candidate_context_size

lowest_count = 0
highest_count = len(balanced_train)
maximum_example_count = 0
working_context_size = 0
calibration_rows = []

while lowest_count <= highest_count:
    example_count = (lowest_count + highest_count) // 2
    full_gpu, context_size = test_example_count(example_count)

    calibration_rows.append({
        "example_count": example_count,
        "context_size": context_size,
        "full_gpu": full_gpu})

    if full_gpu:
        maximum_example_count = example_count
        working_context_size = context_size
        lowest_count = example_count + 1
    else:
        highest_count = example_count - 1

calibration_df = pd.DataFrame(calibration_rows).sort_values("example_count").reset_index(drop=True)

calibration_df

,example_count,context_size,full_gpu
0,252,18122,True
1,378,26774,True
2,441,30817,True
3,457,31777,True
4,461,31979,True
5,462,32062,False
6,463,32099,False
7,465,32181,False
8,473,32777,False
9,505,34735,False


In [8]:
selected_examples = balanced_train.head(maximum_example_count)
many_shot_prefix = build_many_shot_prefix(selected_examples, all_labels)
NUM_CTX = working_context_size

selected_examples["dominant_label"].value_counts()

dominant_label
All-or-nothing thinking    42
Emotional Reasoning        42
Fortune-telling            42
Labeling                   42
Magnification              42
Mental filter              42
Mind Reading               42
No Distortion              42
Overgeneralization         42
Personalization            42
Should statements          41
Name: count, dtype: int64

In [9]:
OLLAMA_URL = OLLAMA_BASE_URL + "/api/generate"

response_schema = {
    "type": "array",
    "items": {
        "type": "string",
        "enum": all_labels}}

def call_ollama(prompt, num_ctx=NUM_CTX):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "format": response_schema,
            "options": {
                "temperature": 0,
                "num_ctx": num_ctx,
                "num_predict": MAX_OUTPUT_TOKENS}},
        timeout=3600)
    return response.json()

def parse_model_answer(raw_answer):
    return json.loads(raw_answer)

In [12]:
def run_many_shot_experiment(test_df, limit=None):
    if limit is None:
        test_part = test_df.copy()
    else:
        test_part = test_df.head(limit).copy()

    results = []
    experiment_start_time = time.time()

    for _, row in test_part.iterrows():
        text = row["text"]
        true_labels = row["labels"]

        prompt = build_test_prompt(many_shot_prefix, text)
        result = call_ollama(prompt)
        raw_answer = result["response"]
        predicted_labels = parse_model_answer(raw_answer)

        results.append({
            "text": text,
            "true_labels": true_labels,
            "predicted_labels": predicted_labels})

    total_time_sec = time.time() - experiment_start_time
    results_df = pd.DataFrame(results)

    return results_df, total_time_sec

results_df, total_time_sec = run_many_shot_experiment(test_df=held_data, limit=None)

results_df

,text,true_labels,predicted_labels
0,The voice are always fimilar (someone she know...,[Personalization],"[Labeling, Mind Reading]"
1,Hello. I have been friend with a guy since gra...,[],"[All-or-nothing thinking, Labeling]"
2,"I thought that he displayed traits of honor, l...",[Labeling],[]
3,"I started going to therapy in December, after ...",[],"[Labeling, Emotional Reasoning]"
4,During this time I was recruited to many great...,[Fortune-telling],[]
...,...,...,...
501,"Hi there, my mother was diagnosed with Bipolar...",[],[]
502,"Lately, I’ve been feeling like someone is watc...",[Emotional Reasoning],"[Mind Reading, Emotional Reasoning]"
503,I have had anxiety almost all of my life but l...,[Mental filter],[]
504,"Hi, for about 3 years now I have been feeling ...",[],"[Emotional Reasoning, Fortune-telling, Labeling]"


In [13]:
metric_names = [
    "All-or-nothing_thinking",
    "Fortune-telling",
    "Mental_filter",
    "Overgeneralization",
    "Labeling",
    "Mind_Reading",
    "Personalization",
    "Emotional_Reasoning",
    "Magnification",
    "Should_statements"]

def llm_evaluator(all_metrics, predictions, text, threshold, min_count):
    predicted_labels = next(predictions)
    predicted_labels = [label.replace(" ", "_") for label in predicted_labels]
    distortions_by_metric = {}

    for metric in all_metrics:
        distortions_by_metric[metric] = metric in predicted_labels

    return distortions_by_metric

predictions = iter(results_df["predicted_labels"])
precision, recall, f1, accuracy = evaluate_tm_df(held_df, predictions, llm_evaluator, 0, metric_names)

metrics_df = pd.DataFrame({
    "metric": metric_names,
    "precision": [precision[metric] for metric in metric_names],
    "recall": [recall[metric] for metric in metric_names],
    "f1": [f1[metric] for metric in metric_names],
    "accuracy": [accuracy[metric] for metric in metric_names]})

macro_f1 = sum(f1.values()) / len(metric_names)

summary_df = pd.DataFrame([{
    "macro_f1": macro_f1,
    "total_time_sec": total_time_sec}])

summary_df

,macro_f1,total_time_sec
0,0.197991,1634.214788


In [14]:
metrics_df

,metric,precision,recall,f1,accuracy
0,All-or-nothing_thinking,0.194444,0.560000,0.288660,0.863636
1,Fortune-telling,0.229885,0.416667,0.296296,0.812253
2,Mental_filter,0.115385,0.400000,0.179104,0.782609
3,Overgeneralization,0.125000,0.084746,0.101010,0.824111
4,Labeling,0.124498,0.756098,0.213793,0.549407
5,Mind_Reading,0.333333,0.234043,0.275000,0.885375
6,Personalization,0.178571,0.113636,0.138889,0.877470
7,Emotional_Reasoning,0.106383,0.285714,0.155039,0.784585
8,Magnification,0.250000,0.169811,0.202247,0.859684
9,Should_statements,0.100000,0.185185,0.129870,0.867589
